# ———————— SMART DEADLINE REMINDER SYSTEM ——————————

## —— Libraries

In [ ]:
import re                  # Pattern matching / regex
import time                # Delay / timer control
import email               # Read email content
import sqlite3             # Local database storage
import datetime as dt      # Date and time handling
import imaplib             # Gmail IMAP connection
import threading           # Background parallel tasks
import tkinter as tk       # Popup GUI alerts
import pandas as pd        # Dataset loading / CSV

from sklearn.feature_extraction.text import TfidfVectorizer   # Text to numerical vectors
from sklearn.linear_model import LogisticRegression           # ML classification model
from sklearn.pipeline import Pipeline                         # ML workflow pipeline

## — CONFIGURATION  (GMAIL) 

In [ ]:
# Creates SQLite database to permanently store tasks and alert status.

USE_GMAIL = True

GMAIL_EMAIL = "ncworkbases@gmail.com"
GMAIL_APP_PASSWORD = "ezcd dcdp wjjc qhio"


emails_manual = [
    "Meeting at 11:45 AM today"
]

## —  DATABASE

In [ ]:
# ============================================================
# DATABASE
# ============================================================

conn = sqlite3.connect("tasks.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tasks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    text TEXT,
    deadline TEXT,
    alerted INTEGER DEFAULT 0
)
""")
conn.commit()

## ——  ML Model

In [ ]:
# Trains Logistic Regression model using dataset to classify important emails.

def train_model():
    df = pd.read_csv("email_dataset_1000.csv")

    model = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(max_iter=1000))
    ])

    model.fit(df["text"], df["label"])
    return model


ml_model = train_model()


def is_important(text):
    return ml_model.predict([text])[0] == 1

## —  TIME PARSER

In [ ]:
# Extracts time/date from email text and converts into proper datetime.

def normalize_time(text):
    match = re.search(r'(\d{1,2})(?::(\d{2}))?\s*(am|pm)?', text.lower())

    if not match:
        return None

    h = int(match.group(1))
    m = int(match.group(2) or 0)
    p = match.group(3)

    if p == "pm" and h != 12:
        h += 12
    if p == "am" and h == 12:
        h = 0

    return h, m


def resolve_date(text):
    if "tomorrow" in text.lower():
        return dt.date.today() + dt.timedelta(days=1)
    return dt.date.today()


def build_datetime(text):
    t = normalize_time(text)
    if not t:
        return None
    return dt.datetime.combine(resolve_date(text), dt.time(*t))

## — EMAIL FETCH

In [ ]:
# Fetches unread Gmail subjects or manual emails depending on mode.

def fetch_gmail():
    inbox = []

    try:
        mail = imaplib.IMAP4_SSL("imap.gmail.com")
        mail.login(GMAIL_EMAIL, GMAIL_APP_PASSWORD)
        mail.select("inbox")

        _, messages = mail.search(None, "UNSEEN")

        for num in messages[0].split():
            _, data = mail.fetch(num, "(RFC822)")

            for part in data:
                if isinstance(part, tuple):
                    msg = email.message_from_bytes(part[1])
                    subject = msg["Subject"]
                    if subject:
                        inbox.append(subject)

        mail.logout()

    except Exception as e:
        print("IMAP ERROR:", e)

    return inbox


def get_emails():
    return fetch_gmail() if USE_GMAIL else emails_manual

## — DATABASE SAVE

In [ ]:
# Saves only unique important tasks and tracks pending alerts.

def save_task(text, deadline):

    cursor.execute(
        "SELECT id FROM tasks WHERE text=? AND deadline=?",
        (text, str(deadline))
    )

    if cursor.fetchone():
        return

    cursor.execute(
        "INSERT INTO tasks (text, deadline, alerted) VALUES (?, ?, 0)",
        (text, str(deadline))
    )

    conn.commit()


def load_tasks():
    cursor.execute("SELECT id, text, deadline FROM tasks WHERE alerted=0")

    tasks = []

    for r in cursor.fetchall():
        tasks.append({
            "id": r[0],
            "text": r[1],
            "deadline": dt.datetime.fromisoformat(r[2])
        })

    return tasks


def mark_alerted(task_id):
    cursor.execute("UPDATE tasks SET alerted=1 WHERE id=?", (task_id,))
    conn.commit()

## — ALERT SYSTEM

In [ ]:
# Shows popup alert window when deadline reminder time arrives.

def show_popup(text):

    window = tk.Tk()
    window.geometry("320x150")
    window.configure(bg="#87CEEB")
    window.overrideredirect(True)

    # =========================================
    # CUSTOM TITLE BAR (DEEP BLUE)
    # =========================================
    title_bar = tk.Frame(window, bg="#0A2A66")
    title_bar.pack(fill="x")

    title_label = tk.Label(
        title_bar,
        text="⏰ ALERT",
        bg="#0A2A66",
        fg="white",
        font=("Segoe UI", 12, "bold")
    )
    title_label.pack(side="left", padx=12, pady=6)

    close_button = tk.Button(
        title_bar,
        text="✖",
        bg="#0A2A66",
        fg="white",
        bd=0,
        activebackground="#08204D",
        activeforeground="white",
        font=("Segoe UI", 11, "bold"),
        command=window.destroy
    )
    close_button.pack(side="right", padx=10)

    # =========================================
    # MAIN ALERT AREA
    # =========================================
    tk.Label(
        window,
        text="—— DEADLINE ALERT ——",
        bg="#87CEEB",
        fg="#0A2A66",
        font=("Segoe UI", 16, "bold")
    ).pack(pady=18)

    tk.Label(
        window,
        text=text,
        bg="#87CEEB",
        fg="black",  # off-black
        font=("Segoe UI", 12, "bold"),
        wraplength=360,
        justify="center"
    ).pack(pady=10)

    # Center window
    window.update_idletasks()
    width = window.winfo_width()
    height = window.winfo_height()
    x = (window.winfo_screenwidth() // 2) - (width // 2)
    y = (window.winfo_screenheight() // 2) - (height // 2)
    window.geometry(f"{width}x{height}+{x}+{y}")

    window.mainloop()

def trigger(task):
    print("ALERT:", task["text"])
    show_popup(task["text"])
    mark_alerted(task["id"])

## — SMART SCHEDULER

In [ ]:
# Filters emails using ML, extracts deadlines, and saves valid tasks.


scheduled = set()

def schedule(task):

    key = task["id"]
    if key in scheduled:
        return

    now = dt.datetime.now()
    deadline = task["deadline"]

    diff = (deadline - now).total_seconds()

    if diff <= 0:
        return

    # YOUR RULES
    if diff > 3 * 3600:
        alert_time = deadline - dt.timedelta(hours=3)

    elif 2 * 3600 < diff <= 3 * 3600:
        alert_time = now + dt.timedelta(minutes=30)

    elif 1 * 3600 < diff <= 2 * 3600:
        alert_time = now + dt.timedelta(minutes=10)

    else:
        alert_time = now + dt.timedelta(minutes=1)

    delay = max((alert_time - now).total_seconds(), 1)

    scheduled.add(key)

    threading.Timer(delay, trigger, args=(task,)).start()

    print("Scheduled:", task["text"], "->", alert_time)

## — PROCESS PIPELINE

In [ ]:
# Filters emails using ML, extracts deadlines, and saves valid tasks.

def process():

    emails = get_emails()

    for text in emails:

        if not is_important(text):
            continue

        deadline = build_datetime(text)

        if deadline:
            save_task(text, deadline)

## — MAIN LOOP

In [ ]:
# Continuously checks emails, schedules tasks, and repeats forever.

def run():

    while True:

        process()

        tasks = load_tasks()

        for t in tasks:
            schedule(t)

        time.sleep(20)

## — START POINT

In [ ]:
# Starts background reminder engine and keeps program alive.

if __name__ == "__main__":

    print("System Started...")

    threading.Thread(target=run, daemon=True).start()

    while True:
        time.sleep(1)